In [73]:
# =========================================================
# PHASE 1 — EMPIRICAL SHIPPING COST FOUNDATION
# Pre-Negotiation Window: Jan 1, 2025 – Oct 31, 2025
# =========================================================
 
# IMPORT LIBRARIES
 
import pandas as pd
import numpy as np
import math
import os
from datetime import datetime
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error


# FORMATTING

# GLOBAL DISPLAY (FOR DEBUGGING ONLY)
pd.options.display.float_format = '{:,.2f}'.format


# EXCEL FORMATTING UTILITIES (USED IN ALL PHASE EXPORTS)
def apply_excel_formatting(workbook, worksheet, df):
    
    currency_format = workbook.add_format({'num_format': '$#,##0.00'})
    percent_format = workbook.add_format({'num_format': '0.00%'})
    number_format  = workbook.add_format({'num_format': '#,##0.00'})
    
    for idx, col in enumerate(df.columns):
        
        col_name = str(col)
        col_width = max(14, len(col_name) + 2)
        
        # CURRENCY DETECTION
        if any(key in col_name.lower() for key in [
            "cost", "revenue", "shipping", "spend", "amount", "premium", "savings", "$"
        ]):
            worksheet.set_column(idx, idx, col_width, currency_format)
        
        # PERCENT DETECTION
        elif any(key in col_name.lower() for key in [
            "rate", "percent", "%"
        ]):
            worksheet.set_column(idx, idx, col_width, percent_format)
        
        # DEFAULT NUMERIC FORMATTING
        else:
            worksheet.set_column(idx, idx, col_width, number_format)

 
# FILE PATHS

order_file = r"[Demand and Fulfilled 2024 and 2025 xlsx file in one workbook, different sheets]"
fedex_file = r"[Fedex Parcel 2024-2025 xlsx file in one workbook, different sheets]"
store_file = r"[Ship Stores xlsx file]"
zip_file = r"[Zip Code master xlsx file]"
ship_company_file = r"[Company shipping xlsx file]"
 
# CREATE OUTPUT FOLDER
output_folder = r"[Desired output folder]"
os.makedirs(output_folder, exist_ok=True)
 
 
# STEP 1A — LOAD & FILTER ORDER DATA (SHIP ONLY)
 
orders_2025 = pd.read_excel(order_file, sheet_name="Fulfilled2025")
 
# SHIP-TO-HOME-ONLY
orders_2025 = orders_2025[orders_2025["Fulfillment Method"] == "STH"]
 
# ONLY SHIPPED LINES
orders_2025 = orders_2025[
    orders_2025["Order Line Status"] == "Shipped"
]
 
# CONVERT SHIPMENT DATE
orders_2025["Shipment Status Ship Date"] = pd.to_datetime(
    orders_2025["Shipment Status Ship Date"],
    format="%m/%d/%Y %I:%M %p",
    errors="coerce"
)
 
# FILTER PRE-NEGOTIATION WINDOW
orders_2025 = orders_2025[
    (orders_2025["Shipment Status Ship Date"] >= "2025-01-01") &
    (orders_2025["Shipment Status Ship Date"] <= "2025-10-31")
]
 
 
# STEP 1B — LOAD COMPANY SHIP CHARGES (TRACKING BRIDGE)
 
ship_company = pd.read_excel(
    ship_company_file
)
 
ship_company["Order Number"] = ship_company["Order Number"].astype(str).str.strip()
ship_company["Tracking Number"] = ship_company["Tracking Number"].astype(str).str.strip()
 
 
# STEP 1C — LOAD & FILTER FEDEX PARCEL DATA
 
fedex = pd.read_excel(
    fedex_file,
    sheet_name="2025 Parcel"
)
 
fedex["Ship Date"] = pd.to_datetime(
    fedex["Ship Date"],
    format="%m/%d/%Y %I:%M %p",
    errors="coerce"
)

# FILTER SAME PRE-NEGOTIATION WINDOW
fedex = fedex[
    (fedex["Ship Date"] >= "2025-01-01") &
    (fedex["Ship Date"] <= "2025-10-31")
]
 
# REMOVE MISSING TRACKING OR ZIP
fedex = fedex[fedex["Shipment Tracking Number"].notna()]
fedex = fedex[fedex["Recipient Postal Code"].notna()]
 
# STANDARDIZE ZIP
fedex["Recipient Postal Code"] = (
    fedex["Recipient Postal Code"].astype(str).str[:5]
)
 
 
# STEP 2A — LOAD STORE LOCATION DATA
 
stores = pd.read_excel(
    store_file,
    sheet_name="Ship Stores"
)

stores = stores[[
    "Store",
    "Store Number",
    "City",
    "State",
    "ZIP",
    "Latitude",
    "Longitude"
]]
 
stores.rename(columns={
    "Store Number": "Store_Number",
    "Store": "Store_Name",
    "ZIP": "Origin_Zip",
    "Latitude": "Origin_Lat",
    "Longitude": "Origin_Lon"
}, inplace=True)
 
stores["Origin_Zip"] = stores["Origin_Zip"].astype(str).str[:5]
 
 
# STEP 2B — LOAD ZIP LAT/LONG REFERENCE
 
zip_df = pd.read_excel(
    zip_file,
    sheet_name="USZipsWithLatLon_20231227"
)
 
zip_df = zip_df[[
    "postal code",
    "place name",
    "admin name1",
    "latitude",
    "longitude"
]]
 
zip_df.rename(columns={
    "postal code": "Destination_Zip",
    "place name": "Destination_City",
    "admin name1": "Destination_State",
    "latitude": "Dest_Lat",
    "longitude": "Dest_Lon"
}, inplace=True)
 
zip_df["Destination_Zip"] = zip_df["Destination_Zip"].astype(str).str[:5]
 
 
# Step 2C — STANDARDIZE ORDER NUMBER FORMAT
 
orders_2025["Order Number"] = (
    orders_2025["Order Number"]
    .astype(str)
    .str.strip()
)
 
ship_company["Order Number"] = (
    ship_company["Order Number"]
    .astype(str)
    .str.strip()
)
 
 
# STEP 2D — MERGE ORDERS WITH COMPANY SHIPPING TRACKING
 
master = pd.merge(
    orders_2025,
    ship_company,
    on="Order Number",
    how="inner"
)

In [74]:
# DIAGNOSTICS TRACKING
 
print("Orders:", len(orders_2025))
print("Company Shipping rows:", len(ship_company))
print("FedEx rows:", len(fedex))
print("Master rows after merge:", len(master))

Orders: 101988
Company Shipping rows: 305337
FedEx rows: 251308
Master rows after merge: 1257213


In [75]:
# STEP 2E — CLEAN TRACKING NUMBERS BEFORE FEDEX MERGE
 
master["Tracking Number"] = (
    master["Tracking Number"]
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str.strip()
)
 
fedex["Shipment Tracking Number"] = (
    fedex["Shipment Tracking Number"]
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str.strip()
)
 
 
# STEP 2F — MERGE FEDEX SHIPPING DATA
 
master = pd.merge(
    master,
    fedex,
    left_on="Tracking Number",
    right_on="Shipment Tracking Number",
    how="inner"
)

In [76]:
# DIAGNOSTICS TRACKING
 
print("Orders:", len(orders_2025))
print("Company Shipping rows:", len(ship_company))
print("FedEx rows:", len(fedex))
print("Master rows after merge:", len(master))

Orders: 101988
Company Shipping rows: 305337
FedEx rows: 251308
Master rows after merge: 1257213


In [77]:
# STEP 2G — ATTACH STORE COORDINATES
 
master = pd.merge(
    master,
    stores,
    left_on="Ship Node Description",
    right_on="Store_Name",
    how="left"
)

In [78]:
# DIAGNOSTICS TRACKING
 
print("Orders:", len(orders_2025))
print("Company Shipping rows:", len(ship_company))
print("FedEx rows:", len(fedex))
print("Master rows after merge:", len(master))

Orders: 101988
Company Shipping rows: 305337
FedEx rows: 251308
Master rows after merge: 1257213


In [79]:
# STEP 2H — ATTACH DESTINATION COORDINATES
 
master = pd.merge(
    master,
    zip_df,
    left_on="Recipient Postal Code",
    right_on="Destination_Zip",
    how="left"
)

In [80]:
# DIAGNOSTICS TRACKING
 
print("Orders:", len(orders_2025))
print("Company Shipping rows:", len(ship_company))
print("FedEx rows:", len(fedex))
print("Master rows after merge:", len(master))

Orders: 101988
Company Shipping rows: 305337
FedEx rows: 251308
Master rows after merge: 1257360


In [81]:
# STEP 3A — CLEAN PRICING ZONE
 
master["Pricing Zone"] = (
    master["Pricing Zone"]
    .astype(str)
    .str.extract("(\d+)", expand=False)
)
 
master["Pricing Zone"] = pd.to_numeric(
    master["Pricing Zone"],
    errors="coerce"
)
 
 
# STEP 3B — CLEAN NUMERIC FIELDS
 
master["Net Charge Amount USD"] = pd.to_numeric(
    master["Net Charge Amount USD"], errors="coerce"
)
 
master["Shipment Rated Weight (Pounds)"] = pd.to_numeric(
    master["Shipment Rated Weight (Pounds)"], errors="coerce"
)
 
master["Original Weight (Pounds)"] = pd.to_numeric(
    master["Original Weight (Pounds)"], errors="coerce"
)

In [82]:
print(master.columns.tolist())

['Order Number_x', 'Order Date', 'SKU', 'SKU Description', 'Prime Line Number', 'Order Line Units Sold', 'Total with tax', 'Total without tax', 'Fulfillment Method', 'Curbside', 'Ship Node Description', 'Customer Name', 'Customer Email', 'Destination ZIP', 'Destination Latitude_x', 'Destination Longitude_x', 'Shipment Status Ship Date', 'Order Line Status', 'Tracking Number', 'Shipment Tracking Number', 'Order Number_y', 'Origin Store', 'Origin ZIP', 'Origin Latitude', 'Origin Longitude', 'Recipient Postal Code', 'Destination Latitude_y', 'Destination Longitude_y', 'Distance Miles', 'Pricing Zone', 'Service Description', 'Ship Date', 'Delivery Date', 'Original Weight (Pounds)', 'Shipment Rated Weight (Pounds)', 'Dimmed Height (in)', 'Dimmed Width (in)', 'Dimmed Length (in)', 'Shipment DIM Flag', 'Net Charge Amount USD', 'Store_Name', 'Store_Number', 'City', 'State', 'Origin_Zip', 'Origin_Lat', 'Origin_Lon', 'Destination_Zip', 'Destination_City', 'Destination_State', 'Dest_Lat', 'Dest_L

In [83]:
# FIX DUPLICATE ORDER NUMBER COLUMNS (AFTER MERGE)

master["Order Number"] = master["Order Number_x"]

master = master.drop(columns=[
    "Order Number_x",
    "Order Number_y"
])

master["Total without tax"] = pd.to_numeric(
    master["Total without tax"], errors="coerce"
)
 
 
# STEP 3C — VOLUME CALCULATION (DIMENSIONS)
 
master["Dimmed Height (in)"] = pd.to_numeric(
    master["Dimmed Height (in)"], errors="coerce"
)
 
master["Dimmed Width (in)"] = pd.to_numeric(
    master["Dimmed Width (in)"], errors="coerce"
)
 
master["Dimmed Length (in)"] = pd.to_numeric(
    master["Dimmed Length (in)"], errors="coerce"
)
 
master["Volume_in3"] = (
    master["Dimmed Height (in)"] *
    master["Dimmed Width (in)"] *
    master["Dimmed Length (in)"]
)
 
 
# STEP 3D — DIM FLAG BINARY
 
master["DIM_Flag"] = master["Shipment DIM Flag"].apply(
    lambda x: 1 if x == "Y" else 0
)
 
 
# STEP 3E — WEIGHT VARIANCE (COMPLIANCE CHECK)
 
master["Weight_Variance"] = (
    master["Shipment Rated Weight (Pounds)"] -
    master["Original Weight (Pounds)"]
)
 
 
# STEP 3F — REVENUE + MARGIN PROXY
 
order_revenue = (
    master.groupby("Order Number")["Total without tax"]
    .sum()
    .reset_index()
    .rename(columns={"Total without tax": "Order_Revenue"})
)
 
master = pd.merge(master, order_revenue, on="Order Number", how="left")
 
master["Margin_Proxy"] = (
    master["Order_Revenue"] -
    master["Net Charge Amount USD"]
)
 
 
# STEP 3G — SKU COUNT PER ORDER (PRIME LINE COUNT)
 
sku_count = (
    master.groupby("Order Number")["Prime Line Number"]
    .nunique()
    .reset_index()
    .rename(columns={"Prime Line Number": "SKU_Count"})
)
 
master = pd.merge(master, sku_count, on="Order Number", how="left")
 
 
# STEP 3H — SHIPPING AS % OF REVENUE
 
master["Shipping_to_Revenue"] = (
    master["Net Charge Amount USD"] /
    master["Order_Revenue"]
)
 
 
# =========================================================
# PHASE 1 COMPLETE
# MASTER DATAFRAME IS NOW MODELING READY
# =========================================================

In [84]:
# =========================================================
# PHASE 2 — DESCRIPTIVE BASELINE (PRE-NEGOTIATION 2025)
# =========================================================
 
 
# STEP 1 — ENSURE CLEAN CORE FIELDS
 
master = master.dropna(subset=[
    "Net Charge Amount USD",
    "Order_Revenue",
    "Pricing Zone",
    "Service Description",
    "Store_Name"
])
 
 
# STEP 2 — EXECUTIVE TOTALS
 
total_shipping_spend = master["Net Charge Amount USD"].sum()
total_orders = master["Order Number"].nunique()
total_shipments = master["Shipment Tracking Number"].nunique()
total_revenue = master["Order_Revenue"].sum()
 
avg_shipping_per_shipment = total_shipping_spend / total_shipments
avg_shipping_per_order = total_shipping_spend / total_orders
shipping_pct_revenue = total_shipping_spend / total_revenue
 
dim_rate = master["DIM_Flag"].mean()
 
executive_summary = pd.DataFrame({
    "Metric": [
        "Total Shipping Spend",
        "Total Revenue",
        "Total Orders",
        "Total Shipments",
        "Avg Shipping per Shipment",
        "Avg Shipping per Order",
        "Shipping as % of Revenue",
        "% Shipments DIM"
    ],
    "Value": [
        total_shipping_spend,
        total_revenue,
        total_orders,
        total_shipments,
        avg_shipping_per_shipment,
        avg_shipping_per_order,
        shipping_pct_revenue,
        dim_rate
    ]
})

In [85]:
# DIAGNOSTICS TRACKING
 
print("Shipping Spend:", total_shipping_spend)
print("Orders:", total_orders)
print("Shipments:", total_shipments)
print("Revenue:", total_revenue)

Shipping Spend: 150718496.45999998
Orders: 25494
Shipments: 251308
Revenue: 114999632978.45003


In [86]:
# STEP 3 — ZONE DISTRIBUTION
 
zone_distribution = (
    master.groupby("Pricing Zone")
    .agg(
        Shipments=("Shipment Tracking Number", "nunique"),
        Avg_Cost=("Net Charge Amount USD", "mean"),
        Total_Cost=("Net Charge Amount USD", "sum")
    )
    .reset_index()
)
 
zone_distribution["% of Shipments"] = (
    zone_distribution["Shipments"] /
    zone_distribution["Shipments"].sum()
)
 
zone_distribution = zone_distribution.sort_values("Pricing Zone")
 
 
# STEP 4 — SERVICE MIX ANALYSIS
 
service_mix = (
    master.groupby("Service Description")
    .agg(
        Shipments=("Shipment Tracking Number", "nunique"),
        Avg_Cost=("Net Charge Amount USD", "mean"),
        Total_Cost=("Net Charge Amount USD", "sum")
    )
    .reset_index()
)
 
service_mix["% of Shipments"] = (
    service_mix["Shipments"] /
    service_mix["Shipments"].sum()
)
 
service_mix = service_mix.sort_values("Total_Cost", ascending=False)
 
 
# STEP 5 — STORE-LEVEL BASELINE
 
store_baseline = (
    master.groupby("Store_Name")
    .agg(
        Shipments=("Shipment Tracking Number", "nunique"),
        Orders=("Order Number", "nunique"),
        Total_Shipping=("Net Charge Amount USD", "sum"),
        Avg_Shipping=("Net Charge Amount USD", "mean"),
        DIM_Rate=("DIM_Flag", "mean"),
        Avg_Weight=("Shipment Rated Weight (Pounds)", "mean"),
        Avg_Zone=("Pricing Zone", "mean"),
        Avg_Shipping_to_Revenue=("Shipping_to_Revenue", "mean")
    )
    .reset_index()
)
 
store_baseline["% of Total Shipping Spend"] = (
    store_baseline["Total_Shipping"] /
    store_baseline["Total_Shipping"].sum()
)
 
store_baseline = store_baseline.sort_values(
    "Total_Shipping", ascending=False
)
 
 
# STEP 6 — SHIPPING COST DISTRIBUTION SUMMARY
 
cost_distribution_summary = master["Net Charge Amount USD"].describe()
 
 
# STEP 7 — MARGIN PROXY SUMMARY
 
margin_summary = master["Margin_Proxy"].describe()
 
 
# =========================================================
# PHASE 2 OUTPUT OBJECTS
# =========================================================
 
# 1. executive_summary
# 2. zone_distribution
# 3. service_mix
# 4. store_baseline
# 5. cost_distribution_summary
# 6. margin_summary
 
 
print("PHASE 2 COMPLETE — Descriptive Baseline Ready")

PHASE 2 COMPLETE — Descriptive Baseline Ready


In [87]:
# =========================================================
# PHASE 3 — STORE-SPECIFIC ZONE COST GRADIENTS
# =========================================================
 
 
# STEP 1 — CLEAN ZONE FIELD
 
master["Pricing Zone"] = pd.to_numeric(
    master["Pricing Zone"], errors="coerce"
)
 
zone_master = master.dropna(subset=[
    "Pricing Zone",
    "Net Charge Amount USD",
    "Store_Name"
])
 
 
# STEP 2 — BUILD STORE x ZONE COST TABLE
 
store_zone_table = (
    zone_master
    .groupby(["Store_Name", "Pricing Zone"])
    .agg(
        Shipments=("Shipment Tracking Number", "nunique"),
        Avg_Shipping_Cost=("Net Charge Amount USD", "mean"),
        Avg_Cost=("Net Charge Amount USD", "mean"),
        Total_Cost=("Net Charge Amount USD", "sum"),
        Avg_Weight=("Shipment Rated Weight (Pounds)", "mean"),
        Avg_Volume=("Volume_in3", "mean")
    )
    .reset_index()
)
 
store_zone_table = store_zone_table.sort_values(
    ["Store_Name", "Pricing Zone"]
)
 
 
# STEP 3 — CALCULATE INCREMENTAL ZONE COST PER STORE
 
# WE COMPUTE THE AVERAGE INCREMENTAL INCREASE PER ZONE STEP
zone_slopes = []
 
for store in store_zone_table["Store_Name"].unique():
    
    temp = store_zone_table[
        store_zone_table["Store_Name"] == store
    ].sort_values("Pricing Zone")
    
    # OONLY COMPUTE SLOPE IF STORE HAS +2 ZONES
    if len(temp) >= 2:
        # LINEAR REGRESSION SLOPE: COST ~ ZONE
        slope = np.polyfit(
            temp["Pricing Zone"],
            temp["Avg_Cost"],
            1
        )[0]
        
        zone_slopes.append({
            "Store_Name": store,
            "Zone_Cost_Slope_$perZone": slope,
            "Min_Zone": temp["Pricing Zone"].min(),
            "Max_Zone": temp["Pricing Zone"].max(),
            "Zone_Span": temp["Pricing Zone"].max() - temp["Pricing Zone"].min()
        })
 
zone_slope_summary = pd.DataFrame(zone_slopes)
 
zone_slope_summary = zone_slope_summary.sort_values(
    "Zone_Cost_Slope_$perZone",
    ascending=False
)
 
 
# STEP 4 — CALCULATE COST PER POUND BY ZONE (PER STORE)
 
store_zone_table["Cost_per_Pound"] = (
    store_zone_table["Total_Cost"] /
    (
        store_zone_table["Shipments"] *
        store_zone_table["Avg_Weight"]
    )
)
 
 
# STEP 5 — OPTIONAL: PIVOT TABLE (VISUALIZATION READY)
 
zone_pivot = store_zone_table.pivot(
    index="Pricing Zone",
    columns="Store_Name",
    values="Avg_Cost"
)
 
 
# STEP 6 — EXPORT PHASE 3 OUTPUT
 
import os
 
phase3_output_path = os.path.join(
    output_folder,
    "Phase3_Store_Specific_Zone_Cost_Tables.xlsx"
)
 
with pd.ExcelWriter(phase3_output_path, engine="xlsxwriter") as writer:
    
    workbook = writer.book
    
    # SHEET 1
    store_zone_table.to_excel(
        writer,
        sheet_name="Store_Zone_Cost_Tables",
        index=False
    )
    ws1 = writer.sheets["Store_Zone_Cost_Tables"]
    apply_excel_formatting(workbook, ws1, store_zone_table)
    
    # SHEET 2
    pivot_table = store_zone_table.pivot(
        index="Store_Name",
        columns="Pricing Zone",
        values="Avg_Shipping_Cost"
    )
    
    pivot_table.columns = [f"Zone {int(col)}" for col in pivot_table.columns]
    
    pivot_table.to_excel(
        writer,
        sheet_name="Pivot_Avg_Cost_By_Zone"
    )
    ws2 = writer.sheets["Pivot_Avg_Cost_By_Zone"]
    apply_excel_formatting(workbook, ws2, pivot_table.reset_index())
 
 
# =========================================================
# PHASE 3 OUTPUT OBJECTS
# =========================================================
 
# 1. store_zone_table  → Detailed zone cost per store
# 2. zone_slope_summary → Incremental $ per zone per store
# 3. zone_pivot → Heatmap-ready matrix
 
 
print("PHASE 3 COMPLETE — 12 Store Zone Gradients Built")
print(f"Saved to: {phase3_output_path}")

PHASE 3 COMPLETE — 12 Store Zone Gradients Built
Saved to: C:\Users\thite\Downloads\PracticeLLC\PracticeLLC_Output_Folder\Phase3_Store_Specific_Zone_Cost_Tables.xlsx


In [88]:
# =========================================================
# PHASE 4 — DIM IMPACT & DIMENSIONAL ECONOMICS
# =========================================================
 
 
# STEP 1 — CLEAN REQUIRED FIELDS
 
dim_master = master.dropna(subset=[
    "DIM_Flag",
    "Net Charge Amount USD",
    "Shipment Rated Weight (Pounds)",
    "Volume_in3"
]).copy()
 
 
# STEP 2 — OVERALL DIM RATE
 
overall_dim_rate = dim_master["DIM_Flag"].mean()
 
 
# STEP 3 — DIM RATE BY STORE
 
dim_by_store = (
    dim_master.groupby("Store_Name")
    .agg(
        Shipments=("Shipment Tracking Number", "nunique"),
        DIM_Rate=("DIM_Flag", "mean"),
        Avg_Cost=("Net Charge Amount USD", "mean")
    )
    .reset_index()
    .sort_values("DIM_Rate", ascending=False)
)
 
 
# STEP 4 — DIM RATE BY SKU
 
dim_by_sku = (
    dim_master.groupby("SKU")
    .agg(
        Shipments=("Shipment Tracking Number", "nunique"),
        DIM_Rate=("DIM_Flag", "mean"),
        Avg_Cost=("Net Charge Amount USD", "mean"),
        Avg_Revenue=("Total without tax", "mean")
    )
    .reset_index()
)
 
# OPTIONAL: ONLY KEEP MEANINGFUL SKUs (MIN VOLUME THRESHOLD)
dim_by_sku = dim_by_sku[dim_by_sku["Shipments"] >= 20]
dim_by_sku = dim_by_sku.sort_values("DIM_Rate", ascending=False)
 
 
# STEP 5 — DIM PREMIUM (CONTROLLED FOR WEIGHT BAND)
 
# CREATE WEIGHT BANDSreate weight bands
dim_master["Weight_Band"] = pd.cut(
    dim_master["Shipment Rated Weight (Pounds)"],
    bins=[0,5,10,20,30,50,100,500],
    right=False
)
 
dim_premium_table = (
    dim_master.groupby(["Weight_Band", "DIM_Flag"], observed=False)
    .agg(
        Avg_Cost=("Net Charge Amount USD", "mean"),
        Shipments=("Shipment Tracking Number", "nunique")
    )
    .reset_index()
)
 
# PIVOT TO CALCULATE PREMIUM
dim_premium_pivot = dim_premium_table.pivot(
    index="Weight_Band",
    columns="DIM_Flag",
    values="Avg_Cost"
)
 
dim_premium_pivot.columns = ["No_DIM_Avg", "DIM_Avg"]
dim_premium_pivot["DIM_Premium_$"] = (
    dim_premium_pivot["DIM_Avg"] -
    dim_premium_pivot["No_DIM_Avg"]
)
 
dim_premium_pivot = dim_premium_pivot.reset_index()
 
 
# STEP 6 — WEIGHT VARIANCE (STORE COMPLIANCE)
 
weight_compliance = (
    dim_master.groupby("Store_Name")
    .agg(
        Avg_Weight_Variance=("Weight_Variance", "mean"),
        Underreported_Rate=("Weight_Variance", lambda x: (x > 0).mean()),
        Avg_Cost=("Net Charge Amount USD", "mean")
    )
    .reset_index()
    .sort_values("Avg_Weight_Variance", ascending=False)
)
 
 
# STEP 7 — VOLUME BREAKPOINT ANALYSIS
 
# CREATE VOLUME BANDS
dim_master["Volume_Band"] = pd.qcut(
    dim_master["Volume_in3"],
    q=10,
    duplicates="drop"
)
 
volume_cost_table = (
    dim_master.groupby("Volume_Band", observed=False)
    .agg(
        Avg_Cost=("Net Charge Amount USD", "mean"),
        Avg_Volume=("Volume_in3", "mean"),
        Shipments=("Shipment Tracking Number", "nunique")
    )
    .reset_index()
)
 
 
# STEP 8 — MARGINAL COST PER CUBIC IN (LINEAR APPROXIMATION)
 
# Simple regression slope: Cost ~ Volume
volume_clean = dim_master.dropna(subset=["Volume_in3"])
 
if len(volume_clean) > 10:
    volume_slope = np.polyfit(
        volume_clean["Volume_in3"],
        volume_clean["Net Charge Amount USD"],
        1
    )[0]
else:
    volume_slope = np.nan
 
 
# STEP 9 — EXPORT PHASE 4 OUTPUT
 
phase4_output_path = os.path.join(
    output_folder,
    "Phase4_DIM_Impact_Analysis.xlsx"
)
 
with pd.ExcelWriter(phase4_output_path, engine="xlsxwriter") as writer:
    
    workbook = writer.book
    
    df1 = pd.DataFrame({"Overall_DIM_Rate": [overall_dim_rate]})
    df1.to_excel(writer, sheet_name="Overall_DIM_Rate", index=False)
    apply_excel_formatting(workbook, writer.sheets["Overall_DIM_Rate"], df1)
    
    dim_by_store.to_excel(writer, sheet_name="DIM_By_Store", index=False)
    apply_excel_formatting(workbook, writer.sheets["DIM_By_Store"], dim_by_store)
    
    dim_by_sku.to_excel(writer, sheet_name="DIM_By_SKU", index=False)
    apply_excel_formatting(workbook, writer.sheets["DIM_By_SKU"], dim_by_sku)
    
    dim_premium_pivot.to_excel(writer, sheet_name="DIM_Premium_By_WeightBand", index=False)
    apply_excel_formatting(workbook, writer.sheets["DIM_Premium_By_WeightBand"], dim_premium_pivot)
    
    weight_compliance.to_excel(writer, sheet_name="Store_Weight_Compliance", index=False)
    apply_excel_formatting(workbook, writer.sheets["Store_Weight_Compliance"], weight_compliance)
    
    volume_cost_table.to_excel(writer, sheet_name="Volume_Breakpoints", index=False)
    apply_excel_formatting(workbook, writer.sheets["Volume_Breakpoints"], volume_cost_table)
    
    df7 = pd.DataFrame({"Volume_Slope_Cost_per_Cubic_IN": [volume_slope]})
    df7.to_excel(writer, sheet_name="Marginal_Volume_Cost", index=False)
    apply_excel_formatting(workbook, writer.sheets["Marginal_Volume_Cost"], df7)
 
 
# =========================================================
# PHASE 4 OUTPUT OBJECTS
# =========================================================
 
# 1. overall_dim_rate
# 2. dim_by_store
# 3. dim_by_sku
# 4. dim_premium_pivot
# 5. weight_compliance
# 6. volume_cost_table
# 7. volume_slope
 
 
print("PHASE 4 COMPLETE — DIM & Dimensional Economics Built")
print(f"Saved to: {phase4_output_path}")

PHASE 4 COMPLETE — DIM & Dimensional Economics Built
Saved to: C:\Users\thite\Downloads\PracticeLLC\PracticeLLC_Output_Folder\Phase4_DIM_Impact_Analysis.xlsx


In [89]:
# =========================================================
# PHASE 5 — FEATURE IMPORTANCE MODEL
# =========================================================
 
 
# STEP 1 — COMPUTE DISTANCE BETWEEN ORIGIN AND DESTINATION
 
def haversine_miles(lat1, lon1, lat2, lon2):
    R = 3958.8
    phi1 = np.radians(lat1)
    phi2 = np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
 
    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlambda/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
 
    return R * c
 
 
master["Distance_Miles"] = haversine_miles(
    master["Origin_Lat"],
    master["Origin_Lon"],
    master["Dest_Lat"],
    master["Dest_Lon"]
)
 
 
# STEP 2 — PREPARE MODELING DATASET
 
model_df = master.copy()
 
# KEEP RELEVANT COLUMNS
model_df = model_df[[
    "Net Charge Amount USD",
    "Shipment Rated Weight (Pounds)",
    "Pricing Zone",
    "Distance_Miles",
    "DIM_Flag",
    "Service Description",
    "Store_Name"
]].dropna()
 
# CONVERT ZONE NUMERIC
model_df["Pricing Zone"] = pd.to_numeric(
    model_df["Pricing Zone"], errors="coerce"
)
 
model_df = model_df.dropna()
 
 
# STEP 3 — CREATE DUMMIES FOR CATEGORICAL VARIABLES
 
model_df = pd.get_dummies(
    model_df,
    columns=["Service Description", "Store_Name"],
    drop_first=True
)
 
 
# STEP 4 — SPLIT FEATURES & TARGET
 
X = model_df.drop(columns=["Net Charge Amount USD"])
y = model_df["Net Charge Amount USD"]
 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
 
 
 
# STEP 5 — FIT LINEAR REGRESSION
 
model = LinearRegression()
model.fit(X_train, y_train)
 
y_pred = model.predict(X_test)
 
 
# STEP 6 — MODEL PERFORMANCE
 
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
 
model_performance = {
    "R2": r2,
    "MAE_$": mae
}
 
 
# STEP 7 — EXTRACT COEFFICIENTS (DOLLAR IMPACT)
 
coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient_$": model.coef_
})
 
coefficients = coefficients.sort_values(
    "Coefficient_$",
    key=abs,
    ascending=False
)
 
intercept = model.intercept_
 
#DISTANCE COEFFICIENT, CONVERTS MILES INTO APPROXIMATE SHIPPING COST EFFECT IN $
distance_coef = coefficients.loc[
    coefficients["Feature"] == "Distance_Miles",
    "Coefficient_$"
].values[0]
 
master["Distance_Cost_Component"] = (
    master["Distance_Miles"] * distance_coef
)
 
 
# STEP 8 — PERCENT CONTRIBUTION (STANDARDIZED APPROACH)
 
# STANDARDIZE FEATURES FOR RELATIVE IMPORTANCE
X_std = (X - X.mean()) / X.std()
model_std = LinearRegression()
model_std.fit(X_std, y)
 
std_coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Std_Coefficient": model_std.coef_
})
 
std_coefficients["Abs_Contribution"] = (
    std_coefficients["Std_Coefficient"].abs()
)
 
std_coefficients["Percent_Contribution"] = (
    std_coefficients["Abs_Contribution"] /
    std_coefficients["Abs_Contribution"].sum()
)
 
std_coefficients = std_coefficients.sort_values(
    "Percent_Contribution",
    ascending=False
)
 
 
# STEP 9 — BUILD EXECUTIVE EQUATION SUMMARY
 
core_drivers = coefficients[
    coefficients["Feature"].isin([
        "Shipment Rated Weight (Pounds)",
        "Pricing Zone",
        "DIM_Flag",
        "Distance_Miles"
    ])
]
 
 
# STEP 10 — EXPORT MODEL OUTPUTS TO EXCEL
 
output_path = os.path.join(output_folder, "Phase_5_Model_Driver_Analysis.xlsx")
 
with pd.ExcelWriter(output_path, engine="xlsxwriter") as writer:
    
    workbook = writer.book
    
    performance_df = pd.DataFrame([model_performance])
    performance_df["Model_Intercept_$"] = intercept
    performance_df.to_excel(writer, sheet_name="Model_Performance", index=False)
    apply_excel_formatting(workbook, writer.sheets["Model_Performance"], performance_df)
    
    coefficients.to_excel(writer, sheet_name="Dollar_Impact_Coefficients", index=False)
    apply_excel_formatting(workbook, writer.sheets["Dollar_Impact_Coefficients"], coefficients)
    
    std_coefficients.to_excel(writer, sheet_name="Percent_Contribution", index=False)
    apply_excel_formatting(workbook, writer.sheets["Percent_Contribution"], std_coefficients)
    
    core_drivers.to_excel(writer, sheet_name="Core_Drivers", index=False)
    apply_excel_formatting(workbook, writer.sheets["Core_Drivers"], core_drivers)
 
 
# =========================================================
# PHASE 5 OUTPUT OBJECTS
# =========================================================
 
# 1. model_performance
# 2. intercept
# 3. coefficients (Dollar impacts)
# 4. std_coefficients (% contributions)
# 5. core_drivers (clean executive view)
 
 
print("PHASE 5 COMPLETE — Feature Importance Model Built")
print(f"File saved to: {output_path}")

PHASE 5 COMPLETE — Feature Importance Model Built
File saved to: C:\Users\thite\Downloads\PracticeLLC\PracticeLLC_Output_Folder\Phase_5_Model_Driver_Analysis.xlsx


In [90]:
# =========================================================
# PHASE 6 — SKU SHIPPING RISK DASHBOARD
# =========================================================
 
 
# STEP 1 — CLEAN DATA
 
sku_master = master.copy()
 
sku_master = sku_master[[
    "SKU",
    "SKU Description",
    "Order Number",
    "Shipment Tracking Number",
    "DIM_Flag",
    "Net Charge Amount USD",
    "Total without tax",
    "Shipment Rated Weight (Pounds)",
    "Volume_in3",
    "Pricing Zone",
    "Store_Name"
]].dropna(subset=["SKU", "Net Charge Amount USD"])
 
 
# STEP 2 — DIM FLAG FIX TO 0 (NO) OR 1 (YES)
 
sku_master["DIM_Flag"] = (
    sku_master["DIM_Flag"]
    .fillna(0)
    .astype(int)
    .clip(0,1)
)
 
 
# STEP 3 — AGGREGATE SKUS
 
sku_summary = (
    sku_master.groupby("SKU")
    .agg(
        SKU_Description=("SKU Description", "first"),
        Total_Shipments=("Shipment Tracking Number", "nunique"),
        DIM_Shipments=("DIM_Flag", "sum"),
        Avg_Cost=("Net Charge Amount USD", "mean"),
        Avg_Revenue=("Total without tax", "mean"),
        Avg_Weight=("Shipment Rated Weight (Pounds)", "mean"),
        Avg_Volume=("Volume_in3", "mean"),
        Avg_Zone=("Pricing Zone", "mean")
    )
    .reset_index()
)
 
 
# STEP 4 — CALCULATE SKU DIM RATE & COST EXPOSURE
 
sku_summary["DIM_Rate"] = (
    sku_summary["DIM_Shipments"] / sku_summary["Total_Shipments"]
)
 
# SHIPPING COST AS % OF REVENUE (MARGIN PROXY)
sku_summary["Shipping_to_Revenue_%"] = (
    sku_summary["Avg_Cost"] / sku_summary["Avg_Revenue"] * 100
)
 
 
# STEP 5 — CLASSIFY SKU RISK BANDS
 
def classify_risk(row):
    if row["DIM_Rate"] >= 0.5 and row["Shipping_to_Revenue_%"] > 15:
        return "High Risk"
    elif row["DIM_Rate"] >= 0.2 or row["Shipping_to_Revenue_%"] > 10:
        return "Medium Risk"
    else:
        return "Low Risk"
 
sku_summary["Risk_Category"] = sku_summary.apply(classify_risk, axis=1)
 
 
# STEP 6 — TOP RISKY SKUS
 
top_risky_skus = sku_summary.sort_values(
    ["Risk_Category", "Shipping_to_Revenue_%", "DIM_Rate"],
    ascending=[False, False, False]
).head(50)
 
 
# STEP 7 — VOLUME / WEIGHT BREAKPOINTS
 
# IDENTIFY TOP 10% BY VOLUME
volume_threshold = sku_summary["Avg_Volume"].quantile(0.90)
weight_threshold = sku_summary["Avg_Weight"].quantile(0.90)
 
high_dim_volume_skus = sku_summary[
    (sku_summary["DIM_Rate"] > 0.3) &
    (sku_summary["Avg_Volume"] >= volume_threshold)
]
 
high_dim_weight_skus = sku_summary[
    (sku_summary["DIM_Rate"] > 0.3) &
    (sku_summary["Avg_Weight"] >= weight_threshold)
]
 
 
# STEP 8 — EXPORT SKU RISK ANALYSIS
 
phase6_path = os.path.join(
    output_folder,
    "Phase_6_SKU_DIM_Risk_Analysis.xlsx"
)
 
with pd.ExcelWriter(phase6_path, engine="xlsxwriter") as writer:
    
    workbook = writer.book
    
    sku_summary.to_excel(writer, sheet_name="SKU_Summary", index=False)
    apply_excel_formatting(workbook, writer.sheets["SKU_Summary"], sku_summary)
    
    top_risky_skus.to_excel(writer, sheet_name="Top_50_Risky_SKUs", index=False)
    apply_excel_formatting(workbook, writer.sheets["Top_50_Risky_SKUs"], top_risky_skus)
    
    high_dim_volume_skus.to_excel(writer, sheet_name="High_Volume_DIM_SKUs", index=False)
    apply_excel_formatting(workbook, writer.sheets["High_Volume_DIM_SKUs"], high_dim_volume_skus)
    
    high_dim_weight_skus.to_excel(writer, sheet_name="High_Weight_DIM_SKUs", index=False)
    apply_excel_formatting(workbook, writer.sheets["High_Weight_DIM_SKUs"], high_dim_weight_skus)
 
 
# =========================================================
# PHASE 6 OUTPUT OBJECTS
# =========================================================
 
# 1. sku_summary → Complete SKU risk metrics
# 2. top_risky_skus → Executive top 50 SKU table
# 3. high_dim_volume_skus → SKU candidates for volume-based packaging improvements
# 4. high_dim_weight_skus → SKU candidates for weight-based packaging/oversize review
 
 
print("PHASE 6 COMPLETE — SKU Shipping Risk Dashboard Built")
print(f"File saved to: {phase6_path}")

PHASE 6 COMPLETE — SKU Shipping Risk Dashboard Built
File saved to: C:\Users\thite\Downloads\PracticeLLC\PracticeLLC_Output_Folder\Phase_6_SKU_DIM_Risk_Analysis.xlsx


In [91]:
# =========================================================
# PHASE 7 — STORE PERFORMANCE SCORECARD
# =========================================================
 
 
# STEP 1 — CLEAN DATA
 
store_master = master.copy()
 
store_master = store_master[[
    "Store_Name",
    "Shipment Tracking Number",
    "DIM_Flag",
    "Net Charge Amount USD",
    "Total without tax",
    "Shipment Rated Weight (Pounds)",
    "Volume_in3",
    "SKU"
]].dropna(subset=["Store_Name", "Shipment Tracking Number"])
 
 
# STEP 2 — AGGREGATE STORE METRICS
 
store_summary = (
    store_master.groupby("Store_Name")
    .agg(
        Total_Shipments=("Shipment Tracking Number", "nunique"),
        DIM_Shipments=("DIM_Flag", "sum"),
        Avg_Cost=("Net Charge Amount USD", "mean"),
        Avg_Revenue=("Total without tax", "mean"),
        Avg_Weight=("Shipment Rated Weight (Pounds)", "mean"),
        Avg_Volume=("Volume_in3", "mean"),
        SKUs_Shipped=("SKU", "nunique")
    )
    .reset_index()
)
 
 
# STEP 3 — DERIVE COMPLIANCE METRICS
 
store_summary["DIM_Rate"] = (
    store_summary["DIM_Shipments"] / store_summary["Total_Shipments"]
)
 
store_summary["Shipping_to_Revenue_%"] = (
    store_summary["Avg_Cost"] / store_summary["Avg_Revenue"] * 100
)
 
# WEIGHT / VOLUME REPORTING DEVIATIONS
store_summary["Weight_Variance_Risk"] = (
    store_master.groupby("Store_Name")["Shipment Rated Weight (Pounds)"]
    .apply(lambda x: x.std() / x.mean())
    .values
)
 
store_summary["Volume_Variance_Risk"] = (
    store_master.groupby("Store_Name")["Volume_in3"]
    .apply(lambda x: x.std() / x.mean())
    .values
)
 
 
# STEP 4 — SCORECARD FORMULA
 
# LOWER DIM RATE + LOWER SHIPPING % OF REVENUE + LOWER VARIANCE = BETTER SCORE
store_summary["Performance_Score"] = (
    (1 - store_summary["DIM_Rate"]) * 0.4 +
    (1 - store_summary["Shipping_to_Revenue_%"]/100) * 0.4 +
    (1 - (store_summary["Weight_Variance_Risk"] + store_summary["Volume_Variance_Risk"])/2) * 0.2
)
 
store_summary = store_summary.sort_values("Performance_Score", ascending=False)
 
 
# STEP 5 — TOP DIM-PROBLEMATIC STORES
 
top_dim_stores = store_summary.sort_values(
    "DIM_Rate",
    ascending=False
).head(10)
 
 
# STEP 6 — EXPORT STORE PERFORMANCE
 
phase7_path = os.path.join(
    output_folder,
    "Phase_7_Store_Performance_Scorecard.xlsx"
)
 
with pd.ExcelWriter(phase7_path, engine="xlsxwriter") as writer:
    
    workbook = writer.book
    
    store_summary.to_excel(writer, sheet_name="Store_Scorecard", index=False)
    apply_excel_formatting(workbook, writer.sheets["Store_Scorecard"], store_summary)
    
    top_dim_stores.to_excel(writer, sheet_name="Top_10_DIM_Stores", index=False)
    apply_excel_formatting(workbook, writer.sheets["Top_10_DIM_Stores"], top_dim_stores)
 
 
# =========================================================
# PHASE 7 OUTPUT OBJECTS
# =========================================================
 
# 1. store_summary → Full performance metrics & scores
# 2. top_dim_stores → Stores with highest DIM issues
 
 
print("PHASE 7 COMPLETE — Store Performance Scorecard Built")
print(f"File saved to: {phase7_path}")

PHASE 7 COMPLETE — Store Performance Scorecard Built
File saved to: C:\Users\thite\Downloads\PracticeLLC\PracticeLLC_Output_Folder\Phase_7_Store_Performance_Scorecard.xlsx


In [92]:
# DIAGNOSTICS BEFORE OPTIMIZATION
 
master.groupby("Pricing Zone").agg(
    shipments=("Order Number","nunique"),
    avg_cost=("Net Charge Amount USD","mean"),
    avg_weight=("Shipment Rated Weight (Pounds)","mean"),
    dim_rate=("DIM_Flag","mean")
).sort_index()

,shipments,avg_cost,avg_weight,dim_rate
Pricing Zone,,,,
2,69,96.07,351.89,0.67
3,532,101.47,364.57,0.71
4,1509,107.00,353.72,0.71
5,5109,111.19,348.81,0.71
6,7756,117.90,354.04,0.71
7,5145,124.45,354.03,0.71
8,5374,132.28,352.45,0.71


In [93]:
# =========================================================
# PHASE 8 — ORIGIN OPTIMIZATION ENGINE
# ========================================================= 
 
# STEP 1 — SETTINGS
top_n_orders = 10000  # TOP N REVENUE ORDERS
 
 
# STEP 2 — FILTER TOP ORDERS
 
top_orders = master.groupby("Order Number")["Order_Revenue"].sum().nlargest(top_n_orders).index
origin_master_filtered = master[master["Order Number"].isin(top_orders)].copy()


# STEP 3 — ENSURE LAT/LON ARE NUMERIC ONCE
origin_master_filtered[['Origin_Lat','Origin_Lon','Dest_Lat','Dest_Lon']] = \
origin_master_filtered[['Origin_Lat','Origin_Lon','Dest_Lat','Dest_Lon']].apply(pd.to_numeric, errors='coerce')

 
# STEP 4 — PREP STORE INFO
 
store_list = master["Store_Name"].unique()
 
# COMPUTE AVERAGE ZONE PER STORE
store_zone_map = master.groupby("Store_Name")["Pricing Zone"].mean().to_dict()
 
# COMPUTE LAT/LON PER STORE (FOR DISTANCE)
store_lat_map = master.groupby("Store_Name")["Origin_Lat"].first().to_dict()
store_lon_map = master.groupby("Store_Name")["Origin_Lon"].first().to_dict()
 
 
# STEP 5 — CORE DRIVERS (MODEL COEFFICIENTS) FROM PHASE 5
 
weight_coef = core_drivers.loc[core_drivers["Feature"]=="Shipment Rated Weight (Pounds)","Coefficient_$"].values[0]
zone_coef = core_drivers.loc[core_drivers["Feature"]=="Pricing Zone","Coefficient_$"].values[0]
dim_coef = core_drivers.loc[core_drivers["Feature"]=="DIM_Flag","Coefficient_$"].values[0]
intercept_val = intercept
 
#DISTANCE COEFFICIENT, CONVERTS MILES INTO APPROXIMATE SHIPPING COST EFFECT IN $
distance_coef = coefficients.loc[coefficients["Feature"] == "Distance_Miles","Coefficient_$"].values[0]
 
 
# STEP 6 — HAVERSINE DISTANCE FUNCTION
 
def haversine_miles(lat1, lon1, lat2, lon2):
    R = 3958.8  # Earth radius in miles
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlambda/2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))
    return R * c
 
 
# STEP 7 — MAP ORDERS TO WEIGHT BANDS (FROM PHASE 4)
 
# DIM_PREMIUM_PIVOT SHOULD HAVE COLUMNS: WEIGHT_BAND, DIM_PREMIUM_$
# MAKE SURE WEIGHT BANDS ARE PD.INTERVALINDEX OBJECTS
weight_bins = dim_premium_pivot["Weight_Band"]
 
def get_dim_penalty(order_weight):
    # FIND THE INTERVAL CONTAINING THE ORDER WEIGHT
    for i, interval in enumerate(weight_bins):
        if order_weight in interval:
            return dim_premium_pivot.loc[i, "DIM_Premium_$"]
    return 0  # FALLBACK IF WEIGHT NOT IN ANY BIN
 
 
# STEP 8 — ORIGIN STORE → DISTANCE → ZONE → COST
 
def distance_to_zone(distance):
    if pd.isna(distance):
        return np.nan
    if distance <= 150:
        return 2
    elif distance <= 300:
        return 3
    elif distance <= 600:
        return 4
    elif distance <= 1000:
        return 5
    elif distance <= 1400:
        return 6
    elif distance <= 1800:
        return 7
    else:
        return 8
 
 
# STEP 9 — SIMULATION
 
simulation_results = []
 
for order_number, group in origin_master_filtered.groupby("Order Number"):
    order_weight = group["Shipment Rated Weight (Pounds)"].sum()
    dim_flag = int(group["DIM_Flag"].any())
 
    # COMPUTE DIM PENALTY FROM WEIGHT BAND
    dim_penalty_dynamic = get_dim_penalty(order_weight) if dim_flag else 0
 
    # DESTINATION LAT/LON
    dest_lat = group["Dest_Lat"].mean()
    dest_lon = group["Dest_Lon"].mean()
    
    # SKIP ORDERS WITH MISSING DESTINATION COORDINATES
    if pd.isna(dest_lat) or pd.isna(dest_lon):
        continue
    
    for store in store_list:
        store_lat = store_lat_map.get(store, np.nan)
        store_lon = store_lon_map.get(store, np.nan)
        
        if pd.isna(store_lat) or pd.isna(store_lon):
            continue
 
        # DISTANCE FROM STORE → DESTINATION        
        distance_miles = haversine_miles(store_lat, store_lon, dest_lat, dest_lon)
 
        # SIMULATED ZONE FROM DISTANCE
        simulated_zone = distance_to_zone(distance_miles)
        
        if pd.isna(simulated_zone):
            continue
        
        pred_cost = (
            intercept_val +
            (weight_coef * order_weight) +
            (zone_coef * simulated_zone) +
            (dim_coef * dim_flag) +
            (distance_coef * distance_miles)
        )
        
        dim_adjusted_cost = pred_cost + dim_penalty_dynamic
        
        simulation_results.append({
            "Order Number": order_number,
            "Sim_Origin_Store": store,
            "Order_Weight": order_weight,
            "Simulated_Zone": simulated_zone,
            "Distance_Miles": distance_miles,
            "Predicted_Shipping_Cost": pred_cost,
            "DIM_Adjusted_Predicted_Cost": dim_adjusted_cost,
            "DIM_Flag": dim_flag,
            "DIM_Penalty_$": dim_penalty_dynamic
        })
 
simulation_df = pd.DataFrame(simulation_results)
 
 
# STEP 10 — FAILSAFE AND INVESTIGATION FOR DISTANCE FAILURES
 
# CHECK FOR NANS
simulation_df = simulation_df.dropna(subset=["DIM_Adjusted_Predicted_Cost"])
bad_distance_orders = simulation_df[simulation_df['Distance_Miles'].isna()]['Order Number'].unique()
if len(bad_distance_orders) > 0:
    print(f"Orders with missing distance: {bad_distance_orders}")

In [94]:
# DIAGNOSTIC INVESTIGATION — NAS FOR DIM_AJUSTED_PREDICTED_COST
 
simulation_df["DIM_Adjusted_Predicted_Cost"].isna().sum()
 
 
# DIAGNOSTIC INVESTIGATION — NAS FOR ORDER NUMBER AND DIM_ADJUSTED_PREDICTED_COST
 
simulation_df.groupby("Order Number")["DIM_Adjusted_Predicted_Cost"].apply(lambda x: x.isna().all()).sum()
 
 
# DIAGNOSTIC INVESTIGATION — COLUMNS FOR SIMULATION
 
simulation_df.columns
 
 
# DIAGNOSTIC INVESTIGATION — IN-SCRIPT LOOK AT NAS TO PREVIEW WORKSHEET FOR NAS FOR DIMS_ADJUSTED_PREDICTED_COST
 
simulation_df[
    simulation_df["DIM_Adjusted_Predicted_Cost"].isna()
].head(20)
 
 
# DIAGNOSTIC INVESTIGATION — SIMULATION IN-SCRIPT LOOK AT NAS TO PREVIEW WORKSHEET FOR ALL NAS IN COLUMNS
 
simulation_df[
    simulation_df["DIM_Adjusted_Predicted_Cost"].isna()
][[
    "Order Number",
    "Order_Weight",
    "DIM_Flag",
    "Simulated_Zone",
    "Distance_Miles"
]].head(20)
 


,Order Number,Order_Weight,DIM_Flag,Simulated_Zone,Distance_Miles


In [95]:
# STEP 11 — IDENTIFY OPTIMAL ORIGIN
optimal_origin = simulation_df.loc[
    simulation_df.groupby("Order Number")["DIM_Adjusted_Predicted_Cost"].idxmin()
].reset_index(drop=True)
 
 
# STEP 12 — MERGE ACTUAL SHIPPING COSTS AND CALCULATE POTENTIAL SAVINGS
 
actual_costs = origin_master_filtered.groupby("Order Number")["Net Charge Amount USD"].sum().reset_index()
actual_costs.rename(columns={"Net Charge Amount USD": "Actual_Shipping_Cost"}, inplace=True)
optimal_origin = optimal_origin.merge(actual_costs, on="Order Number", how="left")
optimal_origin["Potential_Savings_$"] = optimal_origin["Actual_Shipping_Cost"] - optimal_origin["DIM_Adjusted_Predicted_Cost"]
 
 
# STEP 13 — EXPORT TO EXCEL
 
phase8_path = os.path.join(output_folder, "Phase_8_Shipping_Simulation_With_DIM.xlsx")
 
with pd.ExcelWriter(phase8_path, engine="xlsxwriter") as writer:
    
    workbook = writer.book
    
    simulation_df.to_excel(writer, sheet_name="Simulation_Feasible_Stores", index=False)
    apply_excel_formatting(workbook, writer.sheets["Simulation_Feasible_Stores"], simulation_df)
    
    optimal_origin.to_excel(writer, sheet_name="Optimal_Origin", index=False)
    apply_excel_formatting(workbook, writer.sheets["Optimal_Origin"], optimal_origin)
 
 
# =========================================================
# PHASE 8 OUTPUT OBJECTS
# =========================================================
 
# 1. simulation_df → Predicted shipping cost per order per store
# 2. optimal_origin → Optimal store origin per order + predicted cost + savings
# 3. DIM-adjusted cost included for decision-making
 
 
print("PHASE 8 COMPLETE — Origin Optimization Engine Built")
print(f"File saved to: {phase8_path}")

PHASE 8 COMPLETE — Origin Optimization Engine Built
File saved to: C:\Users\thite\Downloads\PracticeLLC\PracticeLLC_Output_Folder\Phase_8_Shipping_Simulation_With_DIM.xlsx
